 # Airport with most delay on average for each carrier (Optimized)

In [1]:
import org.apache.spark

Intitializing Scala interpreter ...

: 

In [ ]:
// DO NOT EXECUTE - this is needed just to avoid showing errors in the following cells
val sc = spark.SparkContext.getOrCreate()

## Parsing the data

Firstly, we need some functions that can parse our CSV files.

In [2]:
def getInt(str:String) : Int = {
    if (str.forall(Character.isDigit))
        str.toInt
    else
        -1
}

def parse_flight(line: String) = {
    val parts = line.split(",")
    val year = getInt(parts(0))
    val month = getInt(parts(1))
    val day = getInt(parts(2))
    val dep_time = parts(4)
    val dep_delay = getInt(parts(15))
    val arr_time = parts(6)
    val arr_delay = getInt(parts(14))
    val carrier = parts(8)
    val flightnum = parts(9)
    val tailnum = parts(10)
    val origin = parts(16)
    val dest = parts(17)
    (year, month, day, dep_time, dep_delay, arr_time, arr_delay, carrier, tailnum, flightnum, origin, dest)
}

def parse_carrier(line: String) = {
    val parts = line.split(",").map(_.trim.replaceAll("^\"|\"$", ""))
    val code = parts(0).toUpperCase
    val description = parts(1)
    (code, description)
}

def parse_airport(line: String) = {
    val parts = line.split(",").map(_.trim.replaceAll("^\"|\"$", ""))
    val iata = parts(0).toUpperCase
    val airport = parts(1)
    val city = parts(2)
    val state = parts(3)
    val country = parts(4)
    val lat = parts(5)
    val long = parts(6)
    (iata, airport, city, state, country, lat, long)
}


getInt: (str: String)Int
parse_flight: (line: String)(Int, Int, Int, String, Int, String, Int, String, String, String, String, String)
parse_carrier: (line: String)(String, String)
parse_airport: (line: String)(String, String, String, String, String, String, String)


## Loading the data

Then we can load our CSV files as RDD's. Then we map with the index to get rid of the CSV file headers and parse flight and aircraft records.

In [ ]:
val flightsPath = "./../../../../datasets/project/flights.csv"
val airportsPath = "./../../../../datasets/project/airports.csv"
val carriersPath = "./../../../../datasets/project/carriers.csv"

val rddFlights = sc.textFile(flightsPath).mapPartitionsWithIndex { (idx, iter) => if (idx == 0) iter.drop(1) else iter }.map(parse_flight)
val rddAirports = sc.textFile(airportsPath).mapPartitionsWithIndex { (idx, iter) => if (idx == 0) iter.drop(1) else iter }.map(parse_airport)
val rddCarriers = sc.textFile(carriersPath).mapPartitionsWithIndex { (idx, iter) => if (idx == 0) iter.drop(1) else iter }.map(parse_carrier)

flightsPath: String = ./../../../../datasets/project/2008.csv/2008.csv
airportsPath: String = ./../../../../datasets/project/airports.csv
carriersPath: String = ./../../../../datasets/project/carriers.csv
rddFlights: org.apache.spark.rdd.RDD[(Int, Int, Int, String, Int, String, Int, String, String, String, String, String)] = MapPartitionsRDD[3] at map at <console>:32
rddAirports: org.apache.spark.rdd.RDD[(String, String, String, String, String, String, String)] = MapPartitionsRDD[7] at map at <console>:33
rddCarriers: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[11] at map at <console>:34


### Step 1: Extract relevant data (carrier, origin, arr_delay)

In [4]:
val carrierAirportDelays = rddFlights
  .map { case (_, _, _, _, _, _, arr_delay, carrier, _, _, origin, _) => 
    ((carrier, origin), (arr_delay, 1)) 
  }
  .coalesce(1)

carrierAirportDelays: org.apache.spark.rdd.RDD[((String, String), (Int, Int))] = CoalescedRDD[13] at coalesce at <console>:29


### Step 2: Calculate average delay per (carrier, origin) pair

In [5]:
val avgDelayPerAirportCarrier = carrierAirportDelays
  .reduceByKey { case ((sumDelay1, count1), (sumDelay2, count2)) =>
    (sumDelay1 + sumDelay2, count1 + count2)
  }
  .mapValues { case (totalDelay, count) => totalDelay.toDouble / count }

avgDelayPerAirportCarrier: org.apache.spark.rdd.RDD[((String, String), Double)] = MapPartitionsRDD[15] at mapValues at <console>:29


### Step 3: Find the airport with the highest delay per carrier

In [6]:
val worstAirportForEachCarrier = avgDelayPerAirportCarrier
  .map { case ((carrier, airport), avgDelay) => (carrier, (airport, avgDelay)) }
  .reduceByKey { case ((airport1, delay1), (airport2, delay2)) =>
    if (delay1 > delay2) (airport1, delay1) else (airport2, delay2)
  }


worstAirportForEachCarrier: org.apache.spark.rdd.RDD[(String, (String, Double))] = ShuffledRDD[17] at reduceByKey at <console>:27


### Step 4: BroadcastJoin with airport and carrier names

In [7]:
val airportMap = rddAirports
  .map(x => (x._1.trim.toUpperCase, x._2.trim))
  .collectAsMap()
val carrierMap = rddCarriers
  .map(x => (x._1.trim.toUpperCase, x._2.trim))
  .collectAsMap()

val airportBroadcast = sc.broadcast(airportMap)
val carrierBroadcast = sc.broadcast(carrierMap)

val results = worstAirportForEachCarrier.map { case (carrier, (airport, avgDelay)) =>
  val airportName = airportBroadcast.value.getOrElse(airport.trim.toUpperCase, "Unknown")
  val carrierName = carrierBroadcast.value.getOrElse(carrier.trim.toUpperCase, "Unknown")
  (s"$carrier ($carrierName)", s"$airport ($airportName)", avgDelay)
}


airportMap: scala.collection.Map[String,String] = Map(CLI -> Clintonville Municipal, DM2 -> Diomede Heliport, SZP -> Santa Paula, M30 -> Metropolis Municipal, CKM -> Fletcher, BRL -> Burlington Municipal, 4B0 -> South Albany, 6V6 -> Hopkins, D87 -> Harbor Springs, CAR -> Caribou Municipal, PPO -> La Porte Municipal, CKV -> Outlaw, C73 -> Dixon Muni-Charles R Walgreen, BUR -> Burbank-Glendale-Pasadena, BTV -> Burlington International, 4B9 -> Simsbury Tri-Town, TTD -> Portland-Troutdale, Q35 -> Springerville Babbitt, LIC -> Limon Municipal, SSI -> Malcolm McKinnon, RED -> Red Lodge, ELP -> El Paso International, DVL -> Devils Lake Municipal-Knoke, DIK -> Dickinson Municipal, T53 -> Robstown-Nueces County, SHR -> Sheridan County, L52 -> Oceano County, S33 -> City-County, 0A8 -> Bibb County...


### Step 5: Collect and print results

In [ ]:
results.saveAsTextFile("output_optimized")

Carrier: B6 (JetBlue Airways), Worst Airport: LGA (LaGuardia), Avg Delay: 30.912190082644628 minutes
Carrier: XE (Expressjet Airlines Inc.), Worst Airport: BGR (Bangor International), Avg Delay: 43.32539682539682 minutes
Carrier: F9 (Frontier Airlines Inc.), Worst Airport: TUL (Tulsa International), Avg Delay: 33.0 minutes
Carrier: FL (AirTran Airways Corporation), Worst Airport: EWR (Newark Intl), Avg Delay: 27.670411985018728 minutes
Carrier: OO (Skywest Airlines Inc.), Worst Airport: ROA (Roanoke Regional/ Woodrum), Avg Delay: 36.554621848739494 minutes
Carrier: AS (Alaska Airlines Inc.), Worst Airport: ORD (Chicago O'Hare International), Avg Delay: 26.985537190082646 minutes
Carrier: YV (Mesa Airlines Inc.), Worst Airport: TVC (Cherry Capital), Avg Delay: 46.84 minutes
Carrier: HA (Hawaiian Airlines Inc.), Worst Airport: SFO (San Francisco International), Avg Delay: 25.553719008264462 minutes
Carrier: UA (United Air Lines Inc.), Worst Airport: MHT (Manchester), Avg Delay: 29.646586